# Per-question certification labels (cache-only replay)

The reproduction runs logged only **totals** for RobustRAG's certificate against one injected passage (k' = 1): Open NQ 25 and RealtimeQA 35 certified at alpha = 0.2. This notebook recovers the **per-question** labels.

- **How:** it runs the authors' own `KeywordAgg.query(..., corruption_size=1)` (pinned commit), exactly as their `main.py` does, but with a stand-in model that returns Mistral's answers from the run's cache. Mistral is never loaded or called; a missing cache entry stops the run.
- **Reads** the two reproduction runs whose paths are hardcoded in cell 3 (alpha = 0.2 only).
- **Writes** `certification-{dataset}-alpha0.2-<timestamp>/certified.json` and `summary.json` to your own output folder.
- **Checks** every recomputed final answer against the run's result file, and the correct and certified totals against the run's log.
- No GPU needed. The setup cells and the last cell are copied from `nq_profiling.ipynb`; only the source-folder wording now names both datasets.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
PROJ = "/content/drive/MyDrive/Fall '26/COSC 89.4"
SHARED = "/content/drive/MyDrive/Fall '26/COSC 89.4/adversarial_rag-shared"
os.chdir(PROJ)
sys.path.insert(0, PROJ)

def run_checked(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

# Colab's Python 3.13 cannot install the authors' 2024 package versions.
# Keep the notebook kernel unchanged and run the experiment in Python 3.11.
run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = Path("/content/robustrag-py311")
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])
SPACY_MODEL = ("https://github.com/explosion/spacy-models/releases/download/"
               "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl")
run_checked([*UV, "pip", "install", "--python", PYTHON,
             "torch==2.2.1", "numpy==1.26.4",
             "transformers==4.40.1", "tokenizers==0.19.1",
             "accelerate==0.29.3", "huggingface-hub==0.23.5",
             "sentencepiece==0.2.0", "joblib==1.3.2",
             "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
             "setuptools<81", SPACY_MODEL])
RUN_ENV = dict(os.environ, NLTK_DATA="/content/robustrag-nltk-data",
               PYTHONHASHSEED="0")
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]
run_checked([PYTHON, "-m", "nltk.downloader", "-d", RUN_ENV["NLTK_DATA"],
             "stopwords"], env=RUN_ENV)
print("Experiment Python:", PYTHON)

## Personal output folder and source run folder

Enter your name when prompted. The notebook is shared, but experiment outputs remain separated as required by `ONBOARDING.md`. The NQ and RealtimeQA reproduction runs are read from the paths hardcoded in `SOURCE_RUNS` (read-only); edit them if your runs live elsewhere.

In [ ]:
from pathlib import Path
import json
import subprocess

OWNER = input("Output folder name (for example, manya or connor): ").strip()
if not OWNER or OWNER.lower() in {"replace-me", "your-name"}:
    raise ValueError("Enter a non-empty personal output folder name.")

DATA_DIR = Path(SHARED) / "data" / "robustrag"
RUNS_DIR = Path(SHARED) / "outputs" / OWNER / "robustrag-reproduction"
SOURCE_RUNS = {  # alpha=0.2 reproduction runs to replay (read-only)
    "open_nq": f"{SHARED}/outputs/connor/robustrag-reproduction/table1-current_v2-open_nq-20261109T021501Z",
    "realtimeqa": f"{SHARED}/outputs/connor/robustrag-reproduction/table1-current_v2-realtimeqa-20261108T232836Z",
}
UPSTREAM_DIR = Path(PROJ) / "third_party" / "robustrag_upstream"
RUNNER_DIR = Path("/content/robustrag-runner")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]
os.environ["ROBUSTRAG_SOURCE_RUNS"] = json.dumps(SOURCE_RUNS)

print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)
for dataset, run_path in SOURCE_RUNS.items():
    print(f"Source run ({dataset}):", run_path, "| found" if Path(run_path).exists() else "| NOT FOUND")

## Prepare the vendored authors' code and verified data

The pinned upstream Python source is committed under `third_party/`, so this step makes no GitHub code download. Shared JSON datasets stay outside git and are downloaded only when missing, with SHA-256 verification.

In [ ]:
from reproduction.robustrag_setup import (
    UPSTREAM_COMMIT, build_runner, ensure_datasets,
)

build_runner(UPSTREAM_DIR, RUNNER_DIR)
ensure_datasets(DATA_DIR)
os.environ["ROBUSTRAG_UPSTREAM_COMMIT"] = UPSTREAM_COMMIT

preflight = """
import json
import spacy, torch, transformers
from nltk.corpus import stopwords
from src.defense import KeywordAgg
from src.models import create_model
assert transformers.__version__ == '4.40.1'
assert 'the' in stopwords.words('english')
assert spacy.load('en_core_web_sm')('Paris')[0].pos_
print(json.dumps({'python': __import__('sys').version,
                  'torch': torch.__version__,
                  'transformers': transformers.__version__}))
"""
preflight_result = subprocess.run(
    [PYTHON, "-c", preflight], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, capture_output=True, text=True,
)
print("Preflight:", preflight_result.stdout.strip())
subprocess.run(
    [PYTHON, RUNNER_DIR / "main.py", "--help"], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, stdout=subprocess.DEVNULL,
)
print("Upstream entry point imports successfully.")

## Certification replay

For each of the first 100 questions of each dataset, the authors' keyword defense is replayed from cache and the certificate's intermediate values, which upstream computes but does not save, are recorded.

How the authors' k' = 1 certificate works (`KeywordAgg.certify`, injection setting):

1. It runs only when the clean defended answer is correct.
2. The bottom-ranked passage is assumed to be replaced by an attacker passage. The threshold becomes `min(beta, alpha * (surviving answering passages + 1))`. If that is at most 1 vote, any keyword could be injected, so the question is not certified.
3. Keywords at or above the threshold are already kept. Keywords exactly one vote short are what one injected response could push through.
4. Every subset of those one-vote-short keywords is added to the kept list, and Mistral's final answer is taken for each. With more than 14 such keywords the code gives up (not certified). Otherwise the question is certified only if every answer is correct; the code stops at the first wrong answer.

`certified.json` fields (one row per question):

| Field | Meaning |
|---|---|
| `final_answer`, `clean_correct` | defended answer with no attack, and whether it contains a gold answer |
| `matches_original_final_answer` | recomputed answer equals the one saved in the run's result file |
| `certified` | certified against one injected passage (only possible when `clean_correct`) |
| `certification_outcome` | `certified`, `wrong_answer_found`, `skipped_too_many_keywords`, `threshold_too_low`, or `not_run_incorrect` |
| `skipped_too_many_keywords` | upstream gave up because more than 14 keywords were one vote short |
| `certify_threshold`, `surviving_answering_passages` | threshold and answering count after the bottom passage is replaced |
| `already_kept_keywords` | keywords that survive without any injection, with `votes` and `source_passage_ranks` (1 = top) |
| `poison_target_candidates` | keywords one vote short, with `votes` and `source_passage_ranks`. Raw: includes gold keywords and fragments |
| `tested_combinations` | each scenario upstream evaluated: `assumed_poisoned_keywords`, the exact `keyword_list_given_to_model`, Mistral's `final_answer`, and `correct` |
| `first_wrong_combination` | the first tested scenario whose answer is incorrect, or `null` |

Keyword lists are `null` when certification did not run (`not_run_incorrect`) or stopped before building them (`threshold_too_low`). `tested_combinations` holds every scenario in the batches upstream evaluated, so it stops near the first wrong answer and is not exhaustive. No passage is ever generated: each scenario edits the keyword list directly.

In [ ]:
%%script env PYTHONHASHSEED=0 /content/robustrag-py311/bin/python
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
import json
import logging
import os
import re
import subprocess
import sys

import joblib

RUNNER_DIR = Path(os.environ["ROBUSTRAG_RUNNER_DIR"])
DATA_DIR = Path(os.environ["ROBUSTRAG_DATA_DIR"])
RUNS_DIR = Path(os.environ["ROBUSTRAG_RUNS_DIR"])
SOURCE_RUNS = {dataset: Path(run) for dataset, run in json.loads(os.environ["ROBUSTRAG_SOURCE_RUNS"]).items()}
UPSTREAM_COMMIT = os.environ["ROBUSTRAG_UPSTREAM_COMMIT"]
if str(RUNNER_DIR) not in sys.path:
    sys.path.insert(0, str(RUNNER_DIR))
import src.defense as defense
from src.dataset_utils import load_data
from src.models import BaseModel
from src.prompt_template import MISTRAL_TMPL

# Python's set iteration order (and so upstream's batch order) depends on the hash seed;
# the reproduction runs used PYTHONHASHSEED=0.
assert os.environ.get("PYTHONHASHSEED") == "0", "Run this cell with PYTHONHASHSEED=0."

DATASETS = {"open_nq": "open_nq.json", "realtimeqa": "realtimeqa.json"}
ALPHA, BETA, CORRUPTION_SIZE, TOP_K = 0.2, 3.0, 1, 10
MAX_ENUMERATED_KEYWORDS = 14  # upstream's "added list too long" limit for short-answer QA
LOG_NAME = "{dataset}-mistral7b-keyword-{alpha}-{beta}-top{top_k}-corr{corruption}-attacknone"


class MissingCacheEntry(RuntimeError):
    pass


class CachedMistral(BaseModel):
    """Returns Mistral's saved answers from the run's cache and never generates."""

    def __init__(self, cache):
        super().__init__()
        self.cache = cache
        self.use_cache = True
        self.prompt_template = MISTRAL_TMPL
        self.batches = []

    def _query(self, prompt):
        if prompt not in self.cache:
            raise MissingCacheEntry(prompt[-400:])
        return self.cache[prompt]

    def _batch_query(self, prompts):
        return [self._query(prompt) for prompt in prompts]

    def batch_query(self, prompts):
        responses = super().batch_query(prompts)
        self.batches.append(list(zip(prompts, responses)))
        return responses


upstream_certify = defense.KeywordAgg.certify


def recording_certify(self, data_item, corruption_size, all_extracted_phrase):
    """Keep upstream's per-passage keyword sets, then run upstream's certify unchanged."""
    self.recorded_phrase_sets = list(all_extracted_phrase)
    return upstream_certify(self, data_item, corruption_size, all_extracted_phrase)


defense.KeywordAgg.certify = recording_certify


class WarningRecorder(logging.Handler):
    def __init__(self):
        super().__init__(logging.WARNING)
        self.messages = []

    def emit(self, record):
        self.messages.append(record.getMessage())


warnings = WarningRecorder()
logging.getLogger("RRAG-main").addHandler(warnings)


def certification_sets(phrase_sets):
    """Recompute upstream certify's intermediate sets (INJECTION branch) to record them."""
    survivors = phrase_sets[:-CORRUPTION_SIZE]
    answering = sum(len(phrases) > 0 for phrases in survivors)
    threshold = min(BETA, ALPHA * (answering + CORRUPTION_SIZE))
    votes, ranks = defaultdict(int), defaultdict(list)
    for rank, phrases in enumerate(survivors, start=1):
        for phrase in phrases:
            votes[phrase] += 1
            ranks[phrase].append(rank)
    votes = {token: count for token, count in votes.items()
             if token not in defense.punctuation and token not in defense.stopword_set}
    kept = [token for token, count in votes.items() if count >= threshold]
    candidates = [token for token, count in votes.items() if threshold - CORRUPTION_SIZE <= count < threshold]
    return answering, threshold, kept, candidates, votes, ranks


def keyword_rows(tokens, votes, ranks):
    ordered = sorted(tokens, key=lambda token: (-votes[token], -len(token), token))
    return [{"keyword": token, "votes": votes[token], "source_passage_ranks": ranks[token]} for token in ordered]


def scenario_prompts(llm, item, kept, candidates):
    """Map each upstream certification prompt to the keywords it assumes were injected."""
    prompts = {}
    for size in range(len(candidates) + 1):
        for combo in combinations(candidates, size):
            tokens = sorted(kept + list(combo), key=lambda token: (len(token), token), reverse=True)
            hints = ", ".join(tokens)
            prompts[llm.wrap_prompt(item, as_multi_choice=False, hints=hints)] = (list(combo), hints)
    return prompts


def find_source_run(dataset):
    run_path = SOURCE_RUNS[dataset]
    if not run_path.name.startswith(f"table1-current_v2-{dataset}-"):
        raise RuntimeError(f"SOURCE_RUNS[{dataset!r}] does not point at a current_v2 {dataset} run: {run_path}")
    manifest_path = run_path / "manifest.json"
    if not manifest_path.exists() or json.loads(manifest_path.read_text()).get("status") != "complete":
        raise RuntimeError(f"Not a complete reproduction run: {run_path}")
    return run_path


def logged_value(log_text, name):
    return int(re.findall(rf"\b{name}: (\d+)", log_text)[-1])  # \b skips "undefended_corr_cnt"


def replay_dataset(dataset):
    run_path = find_source_run(dataset)
    log_name = LOG_NAME.format(dataset=dataset, alpha=ALPHA, beta=BETA, top_k=TOP_K, corruption=CORRUPTION_SIZE)
    log_text = (run_path / "log" / f"{log_name}.log").read_text()
    for setting in (f"alpha={ALPHA},", f"beta={BETA},", f"corruption_size={CORRUPTION_SIZE},", f"top_k={TOP_K},"):
        if setting not in log_text:
            raise RuntimeError(f"{run_path.name} log does not show {setting.rstrip(',')}.")
    original_answers = [entry["defended"] for entry in json.loads((run_path / "result" / f"{log_name}.json").read_text())
                        if "defended" in entry]
    sample_limit = json.loads((run_path / "manifest.json").read_text())["sample_limit"]

    llm = CachedMistral(joblib.load(run_path / "cache" / f"mistral7b-{dataset}-{TOP_K}.z"))
    keyword_agg = defense.KeywordAgg(llm, relative_threshold=ALPHA, absolute_threshold=BETA,
                                     longgen=False, certify_save_path="")
    data_tool = load_data(dataset, TOP_K, data_path=str(DATA_DIR / DATASETS[dataset]))

    rows = []
    for query_index, raw_item in enumerate(data_tool.data[:sample_limit]):
        item = data_tool.process_data_item(raw_item)
        llm.batches, keyword_agg.recorded_phrase_sets = [], None
        warnings.messages.clear()
        try:
            final_answer, certificate = keyword_agg.query(item, corruption_size=CORRUPTION_SIZE)
        except MissingCacheEntry as error:
            raise RuntimeError(f"{dataset} query {query_index} needs an answer missing from the cache; "
                               f"refusing to call the model. Prompt ends with:\n{error}") from None
        clean_correct = data_tool.eval_response(final_answer, item)
        skipped = any("added list too long" in message for message in warnings.messages)

        row = {
            "query_index": query_index, "question": item["question"], "answers": item["answer"],
            "final_answer": final_answer,
            "matches_original_final_answer": final_answer == original_answers[query_index],
            "clean_correct": clean_correct, "certified": bool(clean_correct and certificate),
            "certification_outcome": None, "skipped_too_many_keywords": skipped,
            "certify_threshold": None, "surviving_answering_passages": None,
            "already_kept_keywords": None, "poison_target_candidates": None,
            "tested_combinations": [], "first_wrong_combination": None,
        }
        phrase_sets = keyword_agg.recorded_phrase_sets
        if phrase_sets is None:
            row["certification_outcome"] = "not_run_incorrect"
        else:
            answering, threshold, kept, candidates, votes, ranks = certification_sets(phrase_sets)
            row["certify_threshold"], row["surviving_answering_passages"] = threshold, answering
            if threshold <= CORRUPTION_SIZE:
                row["certification_outcome"] = "threshold_too_low"
            else:
                row["already_kept_keywords"] = keyword_rows(kept, votes, ranks)
                row["poison_target_candidates"] = keyword_rows(candidates, votes, ranks)
                if len(candidates) > MAX_ENUMERATED_KEYWORDS:
                    row["certification_outcome"] = "skipped_too_many_keywords"
                else:
                    scenarios = scenario_prompts(llm, item, kept, candidates)
                    for batch in llm.batches[1:]:  # batch 0 holds the isolated per-passage answers
                        for prompt, response in batch:
                            if prompt not in scenarios:
                                raise RuntimeError(f"{dataset} query {query_index}: a certification prompt "
                                                   "does not match the recomputed scenarios.")
                            assumed, hints = scenarios[prompt]
                            row["tested_combinations"].append({
                                "assumed_poisoned_keywords": assumed, "keyword_list_given_to_model": hints,
                                "final_answer": response, "correct": keyword_agg._eval_response(response, item),
                            })
                    row["first_wrong_combination"] = next(
                        (combo for combo in row["tested_combinations"] if not combo["correct"]), None)
                    row["certification_outcome"] = "certified" if certificate else "wrong_answer_found"

        consistent = (
            (row["certification_outcome"] == "certified") == row["certified"]
            and (row["certification_outcome"] == "skipped_too_many_keywords") == skipped
            and (row["certification_outcome"] != "wrong_answer_found" or row["first_wrong_combination"] is not None)
        )
        if not consistent:
            raise RuntimeError(f"{dataset} query {query_index}: recorded sets disagree with upstream's certificate.")
        rows.append(row)

    mismatches = [row["query_index"] for row in rows if not row["matches_original_final_answer"]]
    summary = {
        "dataset": dataset, "source_run": str(run_path),
        "upstream_commit": UPSTREAM_COMMIT,
        "alpha": ALPHA, "beta": BETA, "corruption_size": CORRUPTION_SIZE, "top_k": TOP_K,
        "sample_size": len(rows), "python_hash_seed": os.environ["PYTHONHASHSEED"],
        "logged_correct": logged_value(log_text, "defended_corr_cnt"),
        "replayed_correct": sum(row["clean_correct"] for row in rows),
        "logged_certified": logged_value(log_text, "certify_cnt"),
        "replayed_certified": sum(row["certified"] for row in rows),
        "final_answer_mismatches": mismatches,
        "outcome_counts": dict(Counter(row["certification_outcome"] for row in rows)),
    }
    summary["checks_passed"] = (not mismatches
                                and summary["logged_correct"] == summary["replayed_correct"]
                                and summary["logged_certified"] == summary["replayed_certified"])

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_dir = RUNS_DIR / f"certification-{dataset}-alpha{ALPHA}-{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "certified.json").write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return output_dir, rows, summary


for dataset in DATASETS:
    output_dir, rows, summary = replay_dataset(dataset)
    print(f"\n=== {dataset} ===")
    print("Saved:", output_dir)
    print(json.dumps({key: value for key, value in summary.items() if key != "source_run"}, indent=2))
    if not summary["checks_passed"]:
        print("WARNING: the replay does not match the original run. "
              "Do not use these labels for attack selection until this is resolved.")
    for row in [row for row in rows if row["certification_outcome"] == "wrong_answer_found"][:3]:
        wrong = row["first_wrong_combination"]
        print(f"\nq{row['query_index']}: {row['question']}  (gold: {row['answers'][:2]})")
        print("  assumed poisoned keywords:", wrong["assumed_poisoned_keywords"])
        print("  keyword list given to model:", wrong["keyword_list_given_to_model"])
        print("  model answered:", wrong["final_answer"])

## Finish

After confirming that all files have finished writing, unmount Drive. Then run `./pull.sh` locally to download the shared outputs. Before committing this notebook, use **Edit → Clear all outputs** so model logs and environment details are not stored in git.

In [ ]:
drive.flush_and_unmount()